# Supply Chain Risk Analysis with NetworkX

This notebook implements supply chain risk analysis using **NetworkX** for graph analytics. This version uses only packages available in the Snowflake conda channel, requiring **no external network access**.

## Why NetworkX?

NetworkX provides pure-Python graph algorithms that work without GPU or external package installation:
- **PageRank**: Identify critical suppliers by network influence
- **Community Detection**: Find supply chain clusters using Louvain algorithm
- **Shortest Paths**: Model risk propagation through the network
- **Centrality Metrics**: Identify bottleneck nodes

## Objectives

1. **Build a supply chain graph** from Snowflake data (Vendors, Materials, Regions)
2. **Compute graph analytics** (PageRank, community detection, centrality)
3. **Identify hidden dependencies** using link prediction heuristics (Jaccard, Adamic-Adar)
4. **Propagate risk scores** through the network
5. **Write results** back to Snowflake tables

## 1. Environment Setup

All packages are sourced from the Snowflake conda channel via `environment.yml`. No pip installation required.

In [ ]:
# =============================================================================
# LIBRARY IMPORTS (All from Snowflake conda channel)
# =============================================================================

import numpy as np
import pandas as pd
from datetime import datetime
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# NetworkX for graph analytics
import networkx as nx
from networkx.algorithms import community as nx_community

# Scikit-learn for ML utilities
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Snowflake session
from snowflake.snowpark.context import get_active_session

print(f"NetworkX version: {nx.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print("[OK] All packages loaded from Snowflake conda channel")

In [ ]:
# Get Snowflake session
session = get_active_session()

# Configuration
ROLE = "SUPPLY_CHAIN_RISK_ROLE"
DATABASE = "SUPPLY_CHAIN_RISK"
SCHEMA = "SUPPLY_CHAIN_RISK"
MODEL_VERSION = "v1.0.0-networkx"

# Set context 
session.sql(f"USE ROLE {ROLE}").collect()
session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()

print(f"[OK] Connected as {ROLE} to {DATABASE}.{SCHEMA}")

## 2. Load Data from Snowflake

In [ ]:
# Load data from Snowflake tables
print("Loading data from Snowflake...")

vendors_df = session.table("VENDORS").to_pandas()
materials_df = session.table("MATERIALS").to_pandas()
regions_df = session.table("REGIONS").to_pandas()
purchase_orders_df = session.table("PURCHASE_ORDERS").to_pandas()
bom_df = session.table("BILL_OF_MATERIALS").to_pandas()
trade_data_df = session.table("TRADE_DATA").to_pandas()

print(f"  Vendors: {len(vendors_df)}")
print(f"  Materials: {len(materials_df)}")
print(f"  Regions: {len(regions_df)}")
print(f"  Purchase Orders: {len(purchase_orders_df)}")
print(f"  Bill of Materials: {len(bom_df)}")
print(f"  Trade Data: {len(trade_data_df)}")
print("[OK] Data loaded")

## 3. Build NetworkX Graph

Create a heterogeneous graph with multiple node types (Vendor, Material, Region) and edge types (SUPPLIES, LOCATED_IN, BOM).

In [ ]:
# =============================================================================
# BUILD NETWORKX GRAPH
# =============================================================================

print("Building NetworkX graph...")

# Create directed graph
G = nx.DiGraph()

# Add vendor nodes with attributes
for _, row in vendors_df.iterrows():
    G.add_node(
        f"V_{row['VENDOR_ID']}", 
        node_type='vendor',
        name=row['NAME'],
        country=row['COUNTRY_CODE'],
        tier=row['TIER'],
        financial_health=row['FINANCIAL_HEALTH_SCORE']
    )

# Add material nodes with attributes
for _, row in materials_df.iterrows():
    G.add_node(
        f"M_{row['MATERIAL_ID']}", 
        node_type='material',
        description=row['DESCRIPTION'],
        criticality=row['CRITICALITY_SCORE'],
        inventory_days=row['INVENTORY_DAYS']
    )

# Add region nodes with attributes
for _, row in regions_df.iterrows():
    G.add_node(
        f"R_{row['REGION_CODE']}", 
        node_type='region',
        name=row['REGION_NAME'],
        base_risk=row['BASE_RISK_SCORE'],
        geopolitical_risk=row['GEOPOLITICAL_RISK'],
        natural_disaster_risk=row['NATURAL_DISASTER_RISK']
    )

print(f"  Nodes added: {G.number_of_nodes()}")

# Add SUPPLIES edges (Vendor -> Material from purchase orders)
supplies_count = 0
for _, row in purchase_orders_df.iterrows():
    src = f"V_{row['VENDOR_ID']}"
    dst = f"M_{row['MATERIAL_ID']}"
    if src in G.nodes and dst in G.nodes:
        weight = float(row['QUANTITY'] * row['UNIT_PRICE'])
        if G.has_edge(src, dst):
            G[src][dst]['weight'] += weight
        else:
            G.add_edge(src, dst, edge_type='SUPPLIES', weight=weight)
            supplies_count += 1

# Add LOCATED_IN edges (Vendor -> Region)
located_count = 0
for _, row in vendors_df.iterrows():
    src = f"V_{row['VENDOR_ID']}"
    dst = f"R_{row['COUNTRY_CODE']}"
    if src in G.nodes and dst in G.nodes:
        G.add_edge(src, dst, edge_type='LOCATED_IN', weight=1.0)
        located_count += 1

# Add BOM edges (Material -> Material)
bom_count = 0
for _, row in bom_df.iterrows():
    src = f"M_{row['PARENT_MATERIAL_ID']}"
    dst = f"M_{row['CHILD_MATERIAL_ID']}"
    if src in G.nodes and dst in G.nodes:
        G.add_edge(src, dst, edge_type='BOM', weight=float(row['QUANTITY_PER_UNIT']))
        bom_count += 1

print(f"  Edges added: {G.number_of_edges()}")
print(f"    - SUPPLIES: {supplies_count}")
print(f"    - LOCATED_IN: {located_count}")
print(f"    - BOM: {bom_count}")
print("[OK] Graph built")

In [ ]:
# =============================================================================
# GRAPH STRUCTURE VISUALIZATION
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Supply Chain Graph Structure', fontsize=14, fontweight='bold')

# 1. Node type distribution
ax = axes[0, 0]
node_types = [G.nodes[n].get('node_type', 'unknown') for n in G.nodes]
type_counts = pd.Series(node_types).value_counts()
colors = {'vendor': '#3498db', 'material': '#e74c3c', 'region': '#27ae60'}
ax.bar(type_counts.index, type_counts.values, color=[colors.get(t, 'gray') for t in type_counts.index])
ax.set_ylabel('Count')
ax.set_title('Node Type Distribution')
for i, (t, c) in enumerate(type_counts.items()):
    ax.text(i, c, str(c), ha='center', va='bottom', fontweight='bold')

# 2. Edge type distribution
ax = axes[0, 1]
edge_types = [G.edges[e].get('edge_type', 'unknown') for e in G.edges]
edge_counts = pd.Series(edge_types).value_counts()
colors_edge = {'SUPPLIES': '#9b59b6', 'LOCATED_IN': '#f39c12', 'BOM': '#1abc9c'}
ax.bar(edge_counts.index, edge_counts.values, color=[colors_edge.get(t, 'gray') for t in edge_counts.index])
ax.set_ylabel('Count')
ax.set_title('Edge Type Distribution')
for i, (t, c) in enumerate(edge_counts.items()):
    ax.text(i, c, str(c), ha='center', va='bottom', fontweight='bold')

# 3. Degree distribution
ax = axes[1, 0]
degrees = [d for n, d in G.degree()]
ax.hist(degrees, bins=30, color='#34495e', edgecolor='white', alpha=0.7)
ax.set_xlabel('Degree')
ax.set_ylabel('Frequency')
ax.set_title(f'Degree Distribution (Mean: {np.mean(degrees):.1f})')
ax.axvline(np.mean(degrees), color='red', linestyle='--', label=f'Mean: {np.mean(degrees):.1f}')
ax.legend()

# 4. Graph summary
ax = axes[1, 1]
summary = {
    'Nodes': G.number_of_nodes(),
    'Edges': G.number_of_edges(),
    'Avg Degree': np.mean(degrees),
    'Density': nx.density(G),
}
ax.bar(summary.keys(), summary.values(), color=['#3498db', '#e74c3c', '#27ae60', '#f39c12'])
ax.set_ylabel('Value')
ax.set_title('Graph Statistics')
for i, (k, v) in enumerate(summary.items()):
    ax.text(i, v, f'{v:.2f}' if isinstance(v, float) else str(v), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()
print("[OK] Graph structure visualized")

## 4. Graph Analytics with NetworkX

Run supply chain analytics using NetworkX algorithms:
- **PageRank**: Identify critical nodes by network influence
- **Betweenness Centrality**: Find bottleneck nodes
- **Community Detection**: Identify supply chain clusters

In [ ]:
# =============================================================================
# PAGERANK: Critical Node Identification
# =============================================================================

print("=" * 70)
print(" PAGERANK: Critical Node Identification ".center(70))
print("=" * 70)

import time
start_time = time.time()
pagerank = nx.pagerank(G, alpha=0.85, max_iter=100, tol=1e-6)
pagerank_time = time.time() - start_time

print(f"\n[OK] PageRank computed in {pagerank_time:.3f}s")

# Convert to DataFrame for analysis
pr_df = pd.DataFrame([
    {'node': k, 'pagerank': v, 'node_type': G.nodes[k].get('node_type', 'unknown')}
    for k, v in pagerank.items()
])

print(f"\nTop 10 Critical Nodes by PageRank:")
print("-" * 50)
for _, row in pr_df.nlargest(10, 'pagerank').iterrows():
    print(f"  {row['node_type']:10s} {row['node']:20s} PR: {row['pagerank']:.6f}")

# Top by node type
print(f"\nTop 5 Critical Vendors:")
vendor_pr = pr_df[pr_df['node_type'] == 'vendor'].nlargest(5, 'pagerank')
for _, row in vendor_pr.iterrows():
    print(f"  {row['node']:25s} PR: {row['pagerank']:.6f}")

print(f"\nTop 5 Critical Materials:")
material_pr = pr_df[pr_df['node_type'] == 'material'].nlargest(5, 'pagerank')
for _, row in material_pr.iterrows():
    print(f"  {row['node']:25s} PR: {row['pagerank']:.6f}")

In [ ]:
# =============================================================================
# BETWEENNESS CENTRALITY: Bottleneck Detection
# =============================================================================

print("=" * 70)
print(" BETWEENNESS CENTRALITY: Bottleneck Detection ".center(70))
print("=" * 70)

start_time = time.time()
# Use approximate betweenness for large graphs
k_samples = min(100, G.number_of_nodes())
betweenness = nx.betweenness_centrality(G, k=k_samples, normalized=True)
betweenness_time = time.time() - start_time

print(f"\n[OK] Betweenness centrality computed in {betweenness_time:.3f}s")

bc_df = pd.DataFrame([
    {'node': k, 'betweenness': v, 'node_type': G.nodes[k].get('node_type', 'unknown')}
    for k, v in betweenness.items()
])

print(f"\nTop 10 Bottleneck Nodes (High Betweenness):")
print("-" * 50)
for _, row in bc_df.nlargest(10, 'betweenness').iterrows():
    print(f"  {row['node_type']:10s} {row['node']:20s} BC: {row['betweenness']:.6f}")

# Vendor bottlenecks
print(f"\nTop 5 Vendor Bottlenecks (Single Points of Failure):")
vendor_bc = bc_df[bc_df['node_type'] == 'vendor'].nlargest(5, 'betweenness')
for _, row in vendor_bc.iterrows():
    print(f"  {row['node']:25s} BC: {row['betweenness']:.6f}")

In [ ]:
# =============================================================================
# COMMUNITY DETECTION: Supply Chain Clusters
# =============================================================================

print("=" * 70)
print(" COMMUNITY DETECTION: Supply Chain Clusters ".center(70))
print("=" * 70)

# Convert to undirected for community detection
G_undirected = G.to_undirected()

start_time = time.time()
# Use Louvain algorithm for community detection
communities = nx_community.louvain_communities(G_undirected, resolution=1.0, seed=42)
community_time = time.time() - start_time

print(f"\n[OK] Community detection completed in {community_time:.3f}s")
print(f"    Communities found: {len(communities)}")

# Create community mapping
node_to_community = {}
for i, comm in enumerate(communities):
    for node in comm:
        node_to_community[node] = i

# Analyze community composition
print(f"\nCommunity Size Distribution:")
print("-" * 60)
for i, comm in enumerate(sorted(communities, key=len, reverse=True)[:10]):
    vendors = sum(1 for n in comm if G.nodes[n].get('node_type') == 'vendor')
    materials = sum(1 for n in comm if G.nodes[n].get('node_type') == 'material')
    regions = sum(1 for n in comm if G.nodes[n].get('node_type') == 'region')
    print(f"  Community {i:3d}: {len(comm):4d} nodes (V:{vendors}, M:{materials}, R:{regions})")

## 5. Link Prediction: Hidden Dependencies

Use graph-based similarity metrics to identify potential hidden relationships between nodes.

In [ ]:
# =============================================================================
# LINK PREDICTION: Jaccard Similarity
# =============================================================================

print("=" * 70)
print(" LINK PREDICTION: Hidden Dependencies ".center(70))
print("=" * 70)

# Get vendor nodes for link prediction (find alternate suppliers)
vendor_nodes = [n for n in G.nodes if G.nodes[n].get('node_type') == 'vendor']

# Compute Jaccard similarity for non-adjacent vendor pairs
start_time = time.time()
jaccard_predictions = []

# Sample pairs to avoid O(n^2) computation
import random
random.seed(42)
sampled_pairs = []
for i, v1 in enumerate(vendor_nodes):
    for v2 in vendor_nodes[i+1:]:
        if not G.has_edge(v1, v2) and not G.has_edge(v2, v1):
            sampled_pairs.append((v1, v2))

# Limit to 1000 pairs for performance
sampled_pairs = random.sample(sampled_pairs, min(1000, len(sampled_pairs)))

# Compute Jaccard coefficient
jaccard_scores = list(nx.jaccard_coefficient(G_undirected, sampled_pairs))
jaccard_time = time.time() - start_time

print(f"\n[OK] Jaccard similarity computed in {jaccard_time:.3f}s")
print(f"    Pairs analyzed: {len(sampled_pairs)}")

# Filter significant predictions
significant_predictions = [(u, v, p) for u, v, p in jaccard_scores if p > 0]
significant_predictions.sort(key=lambda x: x[2], reverse=True)

print(f"\nTop 10 Potential Hidden Links (Alternate Suppliers):")
print("-" * 70)
for u, v, score in significant_predictions[:10]:
    print(f"  {u:20s} <-> {v:20s}  Jaccard: {score:.4f}")

## 6. Risk Propagation

Compute risk scores by propagating regional risk through the supply chain network.

In [ ]:
# =============================================================================
# RISK SCORE COMPUTATION
# =============================================================================

print("=" * 70)
print(" RISK SCORE COMPUTATION ".center(70))
print("=" * 70)

# Find highest risk region
high_risk_region = regions_df.nlargest(1, 'BASE_RISK_SCORE').iloc[0]
source_node = f"R_{high_risk_region['REGION_CODE']}"
print(f"\nHigh-risk source: {high_risk_region['REGION_NAME']} (Risk: {high_risk_region['BASE_RISK_SCORE']:.2f})")

# Compute shortest paths from high-risk region
start_time = time.time()
try:
    distances = nx.single_source_shortest_path_length(G_undirected, source_node)
except nx.NetworkXError:
    # If source not connected, use BFS from all region nodes
    distances = {}
    for region_node in [n for n in G.nodes if G.nodes[n].get('node_type') == 'region']:
        try:
            d = nx.single_source_shortest_path_length(G_undirected, region_node)
            for node, dist in d.items():
                if node not in distances or dist < distances[node]:
                    distances[node] = dist
        except:
            pass

risk_time = time.time() - start_time
print(f"[OK] Risk propagation computed in {risk_time:.3f}s")

# Compute risk scores based on distance and node attributes
risk_scores = {}
for node in G.nodes:
    node_data = G.nodes[node]
    node_type = node_data.get('node_type', 'unknown')
    
    # Base risk from distance to high-risk region
    distance = distances.get(node, float('inf'))
    if distance == float('inf'):
        distance_risk = 0.1  # Low risk if not connected
    else:
        distance_risk = max(0, 1 - (distance * 0.15))  # Decay with distance
    
    # Add node-specific risk factors
    if node_type == 'vendor':
        financial_health = node_data.get('financial_health', 0.5)
        tier = node_data.get('tier', 1)
        node_risk = (1 - financial_health) * 0.3 + (tier / 3) * 0.2
    elif node_type == 'material':
        criticality = node_data.get('criticality', 0.5)
        inventory = node_data.get('inventory_days', 30)
        node_risk = criticality * 0.3 + max(0, (30 - inventory) / 30) * 0.2
    elif node_type == 'region':
        base_risk = node_data.get('base_risk', 0.5)
        geo_risk = node_data.get('geopolitical_risk', 0.5)
        node_risk = (base_risk + geo_risk) / 2
    else:
        node_risk = 0.5
    
    # Combine distance risk and node risk with PageRank importance
    pr = pagerank.get(node, 0)
    combined_risk = distance_risk * 0.4 + node_risk * 0.4 + pr * 100 * 0.2
    risk_scores[node] = min(1.0, combined_risk)

print(f"\nRisk Score Distribution:")
risk_values = list(risk_scores.values())
print(f"  Mean: {np.mean(risk_values):.3f}")
print(f"  Std:  {np.std(risk_values):.3f}")
print(f"  Min:  {np.min(risk_values):.3f}")
print(f"  Max:  {np.max(risk_values):.3f}")

In [ ]:
# =============================================================================
# ANALYTICS VISUALIZATION DASHBOARD
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Supply Chain Risk Analytics Dashboard', fontsize=14, fontweight='bold')

# 1. PageRank distribution
ax = axes[0, 0]
ax.hist(pr_df['pagerank'], bins=50, color='#3498db', edgecolor='white', alpha=0.7)
ax.set_xlabel('PageRank Score')
ax.set_ylabel('Frequency')
ax.set_title(f'PageRank Distribution (computed in {pagerank_time*1000:.1f}ms)')
ax.axvline(pr_df['pagerank'].mean(), color='red', linestyle='--')

# 2. Betweenness distribution
ax = axes[0, 1]
ax.hist(bc_df['betweenness'], bins=50, color='#e74c3c', edgecolor='white', alpha=0.7)
ax.set_xlabel('Betweenness Centrality')
ax.set_ylabel('Frequency')
ax.set_title(f'Betweenness Distribution (computed in {betweenness_time*1000:.1f}ms)')

# 3. Community sizes
ax = axes[1, 0]
comm_sizes = sorted([len(c) for c in communities], reverse=True)[:20]
ax.bar(range(len(comm_sizes)), comm_sizes, color='#9b59b6')
ax.set_xlabel('Community Rank')
ax.set_ylabel('Size (nodes)')
ax.set_title(f'Community Sizes ({len(communities)} communities found)')

# 4. Risk score distribution
ax = axes[1, 1]
ax.hist(risk_values, bins=50, color='#f39c12', edgecolor='white', alpha=0.7)
ax.set_xlabel('Risk Score')
ax.set_ylabel('Frequency')
ax.set_title('Risk Score Distribution')
ax.axvline(np.mean(risk_values), color='red', linestyle='--', label=f'Mean: {np.mean(risk_values):.3f}')
ax.legend()

plt.tight_layout()
plt.show()
print("[OK] Analytics dashboard rendered")

## 7. Write Results to Snowflake

In [ ]:
# =============================================================================
# WRITE RISK SCORES TO SNOWFLAKE
# =============================================================================

print("=" * 70)
print(" WRITING RESULTS TO SNOWFLAKE ".center(70))
print("=" * 70)

# Prepare risk scores dataframe (vendors and materials only)
risk_records = []
timestamp = datetime.now().isoformat()

for node, score in risk_scores.items():
    node_type = G.nodes[node].get('node_type', 'unknown')
    if node_type == 'vendor':
        entity_id = node.replace('V_', '')
        entity_type = 'VENDOR'
    elif node_type == 'material':
        entity_id = node.replace('M_', '')
        entity_type = 'MATERIAL'
    else:
        continue
    
    risk_records.append({
        'ENTITY_TYPE': entity_type,
        'ENTITY_ID': entity_id,
        'RISK_SCORE': float(score),
        'PAGERANK_SCORE': float(pagerank.get(node, 0)),
        'BETWEENNESS_SCORE': float(betweenness.get(node, 0)),
        'COMMUNITY_ID': node_to_community.get(node, -1),
        'MODEL_VERSION': MODEL_VERSION,
        'COMPUTED_AT': timestamp
    })

risk_df = pd.DataFrame(risk_records)

# Write to RISK_SCORES table
session.sql("TRUNCATE TABLE IF EXISTS NX_RISK_SCORES").collect()
session.write_pandas(risk_df, "NX_RISK_SCORES", auto_create_table=False, overwrite=False)
print(f"[OK] Wrote {len(risk_df)} risk scores to NX_RISK_SCORES table")

In [ ]:
# =============================================================================
# WRITE PREDICTED LINKS TO SNOWFLAKE
# =============================================================================

# Prepare predicted links dataframe
link_records = []
for source, target, score in significant_predictions[:100]:
    source_id = source.replace('V_', '').replace('M_', '').replace('R_', '')
    target_id = target.replace('V_', '').replace('M_', '').replace('R_', '')
    source_type = G.nodes[source].get('node_type', 'unknown').upper()
    target_type = G.nodes[target].get('node_type', 'unknown').upper()
    
    link_records.append({
        'SOURCE_TYPE': source_type,
        'SOURCE_ID': source_id,
        'TARGET_TYPE': target_type,
        'TARGET_ID': target_id,
        'SIMILARITY_SCORE': float(score),
        'PREDICTION_METHOD': 'JACCARD',
        'MODEL_VERSION': MODEL_VERSION,
        'PREDICTED_AT': timestamp
    })

links_df = pd.DataFrame(link_records)

# Write to PREDICTED_LINKS table
session.sql("TRUNCATE TABLE IF EXISTS NX_PREDICTED_LINKS").collect()
session.write_pandas(links_df, "NX_PREDICTED_LINKS", auto_create_table=False, overwrite=False)
print(f"[OK] Wrote {len(links_df)} predicted links to NX_PREDICTED_LINKS table")

In [ ]:
# =============================================================================
# WRITE BOTTLENECKS TO SNOWFLAKE
# =============================================================================

# Get top bottleneck nodes
bottleneck_records = []
for _, row in bc_df.nlargest(50, 'betweenness').iterrows():
    node = row['node']
    node_type = row['node_type'].upper()
    entity_id = node.replace('V_', '').replace('M_', '').replace('R_', '')
    
    bottleneck_records.append({
        'ENTITY_TYPE': node_type,
        'ENTITY_ID': entity_id,
        'BETWEENNESS_SCORE': float(row['betweenness']),
        'PAGERANK_SCORE': float(pagerank.get(node, 0)),
        'RISK_SCORE': float(risk_scores.get(node, 0)),
        'COMMUNITY_ID': node_to_community.get(node, -1),
        'BOTTLENECK_RANK': _ + 1,
        'MODEL_VERSION': MODEL_VERSION,
        'IDENTIFIED_AT': timestamp
    })

bottlenecks_df = pd.DataFrame(bottleneck_records)

# Write to BOTTLENECKS table
session.sql("TRUNCATE TABLE IF EXISTS NX_BOTTLENECKS").collect()
session.write_pandas(bottlenecks_df, "NX_BOTTLENECKS", auto_create_table=False, overwrite=False)
print(f"[OK] Wrote {len(bottlenecks_df)} bottlenecks to NX_BOTTLENECKS table")

## 8. Results Summary

In [ ]:
# =============================================================================
# EXECUTION SUMMARY
# =============================================================================

print("=" * 70)
print(" SUPPLY CHAIN RISK ANALYSIS COMPLETE ".center(70))
print("=" * 70)

print(f"""
GRAPH STATISTICS:
  - Nodes: {G.number_of_nodes():,}
  - Edges: {G.number_of_edges():,}
  - Density: {nx.density(G):.4f}

ANALYTICS RESULTS:
  - Communities detected: {len(communities)}
  - Avg PageRank: {pr_df['pagerank'].mean():.6f}
  - Avg Betweenness: {bc_df['betweenness'].mean():.6f}
  - Avg Risk Score: {np.mean(risk_values):.3f}

TABLES UPDATED:
  - RISK_SCORES: {len(risk_df)} records
  - PREDICTED_LINKS: {len(links_df)} records
  - BOTTLENECKS: {len(bottlenecks_df)} records

MODEL VERSION: {MODEL_VERSION}
TIMESTAMP: {timestamp}
FRAMEWORK: NetworkX (no external network access required)
""")

# Verify written data
verify_risk = session.sql("SELECT COUNT(*) as cnt FROM NX_RISK_SCORES").collect()[0]['CNT']
verify_links = session.sql("SELECT COUNT(*) as cnt FROM NX_PREDICTED_LINKS").collect()[0]['CNT']
verify_bottlenecks = session.sql("SELECT COUNT(*) as cnt FROM NX_BOTTLENECKS").collect()[0]['CNT']

print("DATA VERIFICATION:")
print(f"  RISK_SCORES: {verify_risk} records")
print(f"  PREDICTED_LINKS: {verify_links} records")
print(f"  BOTTLENECKS: {verify_bottlenecks} records")
print("\n[OK] All results verified in Snowflake")